# Méta-classes

## Modification de méthodes

Écrivez une méta-classe `JournaliseCalculsMeta` qui transforme toutes les méthodes ordinaires d'une classe (ni `@staticmethod`, ni `@classmethod`) dont le nom contient le mot "calcule" avec le décorateur `journalise_appels` de la cellule suivante, qui journalise les appels aux méthodes qu'il décore. Un attribut qui n'est pas une méthode, comme `mode_calcule` ci-dessous, ne change pas.

Avec cette classe :

```python
class A(metaclass=JournaliseCalculsMeta):
  mode_calcule = "exact"

  def calcule_addition(self, a, b):
    return a + b


a = A()
a.calcule_addition(1, 2)
print(a.mode_calcule)
```

Vous devez obtenir :

```text
calcule_addition appelée avec les args (1, 2) et kwargs {} a calculé 3
exact
```

**Documentation utile**

- [`inspect.isfunction`](https://docs.python.org/fr/3/library/inspect.html#inspect.isfunction)

In [ ]:
import functools


def journalise_appels(f):

  @functools.wraps(f)
  def enveloppe(self, *args, **kwargs):
    resultat = f(self, *args, **kwargs)
    print(
      f"{f.__name__} appelée avec les args {args} et kwargs {kwargs} a calculé {resultat}"
    )
    return resultat

  return enveloppe

In [ ]:
# Votre code ici

### Conseils supplémentaires

`__new__` reçoit dans `espace_noms` le dictionnaire des noms définis dans le corps de la classe. Construisez-en un nouveau, où les méthodes visées passent par `journalise_appels`, et donnez-le à `super().__new__`. `inspect.isfunction(v)` vaut `True` pour une méthode ordinaire et `False` pour la chaîne de `mode_calcule`.

```python
import inspect


class JournaliseCalculsMeta(type):
  def __new__(mcs, nom, parents, espace_noms):
    nouvel_espace = {
      k: (journalise_appels(v) if ... else v) for k, v in espace_noms.items()
    }
    return super().__new__(mcs, nom, parents, nouvel_espace)
```

### Solution

In [ ]:
import inspect


class JournaliseCalculsMeta(type):
  def __new__(mcs, nom, parents, espace_noms):
    # Dans l'espace de noms, staticmethod et classmethod ne sont pas des fonctions :
    # isfunction les laisse de côté, comme mode_calcule. Avec callable, un staticmethod,
    # appelable depuis Python 3.10, passerait par journalise_appels et recevrait un self en trop
    nouvel_espace = {
      k: (journalise_appels(v) if "calcule" in k and inspect.isfunction(v) else v)
      for k, v in espace_noms.items()
    }
    return super().__new__(mcs, nom, parents, nouvel_espace)


class A(metaclass=JournaliseCalculsMeta):
  mode_calcule = "exact"

  def calcule_addition(self, a, b):
    return a + b


a = A()
a.calcule_addition(1, 2)
# mode_calcule n'est pas une fonction, il reste une chaîne
print(a.mode_calcule)

## Valider les sous-classes : méta-classe ou `__init_subclass__`

On veut que toute sous-classe d'`Exportateur` définisse un attribut de classe `extension`, une chaîne commençant par un point, et qu'une `TypeError` soit levée dès la définition de la classe sinon.

1. Implémentez cette vérification avec une méta-classe `ExportateurMeta`.
2. Implémentez-la ensuite sans méta-classe, avec `__init_subclass__`.

Dans les deux cas, `class ExportateurCsv(Exportateur): extension = ".csv"` doit être acceptée, et `class ExportateurCasse(Exportateur): pass` doit lever une `TypeError`.

**Documentation utile**

- [Méta-classes](https://docs.python.org/fr/3/reference/datamodel.html#metaclasses)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Le `__init__` d'une méta-classe reçoit la classe déjà créée : appelez d'abord `super().__init__(nom, parents, espace_noms)`, puis vérifiez. `if parents` écarte `Exportateur` elle-même, qui n'a pas de parents ; `__init_subclass__` n'en a pas besoin, car elle n'est appelée que pour les sous-classes. `getattr(cls, "extension", None)` lit l'attribut, hérité ou non, sans erreur s'il manque, et `isinstance(extension, str)` vérifie que c'est une chaîne avant d'appeler `startswith`.

À vérifier : `extension = ".csv"` passe ; une classe sans `extension`, `extension = "csv"` et `extension = 1` lèvent une `TypeError`.

```python
class ExportateurMeta(type):
  def __init__(cls, nom, parents, espace_noms):
    super().__init__(nom, parents, espace_noms)
    extension = getattr(cls, "extension", None)
    if parents and not ...:
      raise TypeError(f"{nom} doit définir une extension commençant par un point")


class Exportateur(metaclass=ExportateurMeta):
  pass


# Sans méta-classe
class Exportateur:
  def __init_subclass__(cls, **kwargs):
    super().__init_subclass__(**kwargs)
    extension = getattr(cls, "extension", None)
    ...
```

### Solution

In [ ]:
import abc


class ExportateurMeta(type):
  def __init__(cls, nom, parents, espace_noms):
    super().__init__(nom, parents, espace_noms)
    extension = getattr(cls, "extension", None)
    if parents and not (isinstance(extension, str) and extension.startswith(".")):
      raise TypeError(f"{nom} doit définir une extension commençant par un point")


class Exportateur(metaclass=ExportateurMeta):
  pass


class ExportateurCsv(Exportateur):
  extension = ".csv"


try:

  class ExportateurCasse(Exportateur):
    pass

except TypeError as erreur:
  print(erreur)

try:

  class ExportateurAbstrait(Exportateur, abc.ABC):
    extension = ".txt"

except TypeError as erreur:
  print(erreur)


# Sans méta-classe
class Exportateur:
  def __init_subclass__(cls, **kwargs):
    super().__init_subclass__(**kwargs)
    extension = getattr(cls, "extension", None)
    if not (isinstance(extension, str) and extension.startswith(".")):
      raise TypeError(
        f"{cls.__name__} doit définir une extension commençant par un point"
      )


class ExportateurCsv(Exportateur):
  extension = ".csv"


try:

  class ExportateurCasse(Exportateur):
    pass

except TypeError as erreur:
  print(erreur)


class ExportateurAbstrait(Exportateur, abc.ABC):
  extension = ".txt"


print(type(ExportateurAbstrait))

`__init_subclass__` donne le même résultat sans méta-classe : le code est plus simple, et la classe reste combinable avec d'autres classes qui ont leur propre méta-classe. `ExportateurAbstrait`, qui hérite aussi d'`abc.ABC`, provoque un conflit de méta-classes avec `ExportateurMeta`, et passe avec `__init_subclass__`.

## Implémentation du pattern singleton

Le pattern singleton consiste à renvoyer toujours la même instance quand du code client demande une nouvelle instance d'une classe.

Créez une méta-classe `SingletonMeta` qui implémente ce pattern. Attention, notez bien qu'une méta-classe peut être appliquée à plusieurs classes : chacune a sa propre instance.

Une méta-classe peut réimplémenter plusieurs méthodes. Ici, le plus simple sera de redéfinir `__call__`.

**Documentation utile**

- [`object.__call__`](https://docs.python.org/fr/3/reference/datamodel.html#object.__call__)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Le `__call__` de la méta-classe reçoit `cls`, la classe dont le code client demande une instance, et `super().__call__(*args, **kwargs)` crée cette instance (il appelle `__new__` et `__init__` de `cls`). Gardez les instances dans un dictionnaire dont les clés sont les classes.

À vérifier avec deux classes `A` et `B` : `A() is A()` vaut `True`, `A() is B()` vaut `False`, et le `__init__` de `A` ne s'exécute qu'une fois.

```python
class SingletonMeta(type):
  _instances = {}

  def __call__(cls, *args, **kwargs):
    if cls not in SingletonMeta._instances:
      SingletonMeta._instances[cls] = ...
    return ...
```

### Solution

In [ ]:
class SingletonMeta(type):
  _instances = {}

  def __call__(cls, *args, **kwargs):
    if cls not in SingletonMeta._instances:
      SingletonMeta._instances[cls] = super().__call__(*args, **kwargs)
    return SingletonMeta._instances[cls]


class A(metaclass=SingletonMeta):
  def __init__(self):
    print("Création de l'instance de A")


class B(metaclass=SingletonMeta):
  pass


print(A() is A())
print(A() is B())

Trois limites à connaître avant d'utiliser cette solution :

- **Un module suffit souvent.** Python n'exécute un module qu'une fois : un objet créé au niveau du module est déjà unique, sans méta-classe.
- **`__new__` à la place de `__call__`.** Un singleton écrit avec le `__new__` de la classe relance `__init__` à chaque appel, car `type.__call__` appelle `__init__` sur l'objet que `__new__` renvoie. Avec le `__call__` de la méta-classe, `__init__` ne s'exécute qu'une fois.
- **Fils d'exécution.** Plusieurs fils peuvent passer le test `cls not in ...` avant que la première instance existe, et chacun crée la sienne. La cellule suivante le montre avec quatre fils et un `__init__` lent, puis corrige avec un `threading.Lock`.

In [ ]:
import threading
import time


def compter_instances(classe):
  objets = []
  fils = [threading.Thread(target=lambda: objets.append(classe())) for _ in range(4)]
  for fil in fils:
    fil.start()
  for fil in fils:
    fil.join()
  return len({id(objet) for objet in objets})


class Lente(metaclass=SingletonMeta):
  def __init__(self):
    time.sleep(0.1)


print(compter_instances(Lente))


class SingletonVerrouMeta(type):
  _instances = {}
  _verrou = threading.Lock()

  def __call__(cls, *args, **kwargs):
    with SingletonVerrouMeta._verrou:
      if cls not in SingletonVerrouMeta._instances:
        SingletonVerrouMeta._instances[cls] = super().__call__(*args, **kwargs)
    return SingletonVerrouMeta._instances[cls]


class LenteVerrou(metaclass=SingletonVerrouMeta):
  def __init__(self):
    time.sleep(0.1)


print(compter_instances(LenteVerrou))

## Décorateur de classe ou méta-classe

Un décorateur de classe suffit souvent à la place d'une méta-classe. Comparez les deux.

1. Écrivez le décorateur de classe `journaliser_calculs`, qui fait le même travail que `JournaliseCalculsMeta` (premier exercice) sur la classe qu'il décore.
2. Appliquez-le à une classe `Addition` qui a une méthode `calcule_addition`, puis écrivez une sous-classe `Produit` qui ajoute une méthode `calcule_produit`.
3. Écrivez les mêmes classes avec la méta-classe : `AdditionMeta`, avec `metaclass=JournaliseCalculsMeta`, et sa sous-classe `ProduitMeta`.
4. Appelez `calcule_produit` sur une instance de `Produit` et sur une instance de `ProduitMeta`. Quel appel est journalisé, et pourquoi ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Un décorateur de classe reçoit la classe et la renvoie. `vars(cls)` donne ses attributs, et `setattr(cls, nom, valeur)` en remplace un. Parcourez une copie, `list(vars(cls).items())`, plutôt que le dictionnaire que `setattr` modifie.

```python
import inspect


def journaliser_calculs(cls):
  for nom, valeur in list(vars(cls).items()):
    if ...:
      setattr(cls, nom, ...)
  return cls
```

### Solution

In [ ]:
import inspect


def journaliser_calculs(cls):
  for nom, valeur in list(vars(cls).items()):
    if "calcule" in nom and inspect.isfunction(valeur):
      setattr(cls, nom, journalise_appels(valeur))
  return cls


@journaliser_calculs
class Addition:
  def calcule_addition(self, a, b):
    return a + b


class Produit(Addition):
  def calcule_produit(self, a, b):
    return a * b


class AdditionMeta(metaclass=JournaliseCalculsMeta):
  def calcule_addition(self, a, b):
    return a + b


class ProduitMeta(AdditionMeta):
  def calcule_produit(self, a, b):
    return a * b


print("Décorateur :")
Produit().calcule_addition(1, 2)
Produit().calcule_produit(2, 3)
print("Méta-classe :")
ProduitMeta().calcule_addition(1, 2)
ProduitMeta().calcule_produit(2, 3)

Seule la méta-classe journalise `calcule_produit`. Le décorateur s'applique une fois, à la classe qu'il décore : `Produit` hérite de `calcule_addition` déjà décorée, mais sa nouvelle méthode n'y passe pas. Une méta-classe s'hérite : `type(ProduitMeta)` est `JournaliseCalculsMeta`, dont le `__new__` s'exécute aussi pour les sous-classes. `__init_subclass__`, qui s'hérite lui aussi, donnerait ce comportement sans méta-classe.